# Generate cradle-to-gate ethylene routes

Creates the Brightway activities for the fossil, H2, and eCO2R ethylene routes used by the later optimex case-study notebook. The system boundary ends at 1 kg produced ethylene after final product separation; use phase and product incineration are not modelled.

In [1]:
from pathlib import Path
import sys

import bw2data as bd
import pandas as pd

PROJECT = "optimex_remind"
ECOINVENT_DB = "ecoinvent-3.12-cutoff"
BIOSPHERE_DB = "ecoinvent-3.12-biosphere"
CUSTOM_DB = "disco2very"

DATA_DIR = Path.cwd()
if not (DATA_DIR / "my_activities.py").exists():
    DATA_DIR = Path("notebooks/data/disco2very_data").resolve()
if str(DATA_DIR) not in sys.path:
    sys.path.insert(0, str(DATA_DIR))

from my_activities import MyActivities

bd.projects.set_current(PROJECT)
missing = [name for name in [ECOINVENT_DB, BIOSPHERE_DB, CUSTOM_DB] if name not in bd.databases]
if missing:
    raise RuntimeError(
        f"Missing database(s) in Brightway project {PROJECT!r}: {missing}. "
        "Run setup_auxiliary_activities.ipynb first and make sure ecoinvent is imported."
    )

eidb = bd.Database(ECOINVENT_DB)
ma = MyActivities(project=PROJECT, ecoinvent_db=ECOINVENT_DB, biosphere_db=BIOSPHERE_DB, custom_db=CUSTOM_DB)
rows = []
blockers = []

c:\Users\lucal\Brightway\optimex\.venv\Lib\site-packages\bw2calc\__init__.py:57: UserWarning: No fast sparse solver found
  warnings.warn("No fast sparse solver found")
c:\Users\lucal\Brightway\optimex\.venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.2) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


In [2]:
def value(activity, key):
    try:
        return activity.get(key)
    except Exception:
        return None


def add_step(route, step, activity, status="modelled"):
    rows.append({
        "route": route,
        "step": step,
        "activity name": value(activity, "name"),
        "code": activity.key[1],
        "database": activity.key[0],
        "location": value(activity, "location"),
        "unit": value(activity, "unit"),
        "reference product": value(activity, "reference product"),
        "modelling status": status,
    })


def add_blocker(route, step, message):
    blockers.append({"route": route, "step": step, "blocker": message})
    rows.append({
        "route": route,
        "step": step,
        "activity name": None,
        "code": None,
        "database": None,
        "location": None,
        "unit": None,
        "reference product": None,
        "modelling status": f"BLOCKED: {message}",
    })

## Fossil route

In [3]:
try:
    eth_steam_crack = eidb.get(
        name="unsaturated hydrocarbons production, steam cracking operation, average",
        location="RER w/o RU",
        product="ethylene",
    )
    add_step(
        "Fossil route",
        "Steam cracking, cradle-to-gate ethylene",
        eth_steam_crack,
        "existing ecoinvent activity; endpoint is 1 kg produced ethylene",
    )
except Exception as exc:
    add_blocker("Fossil route", "Steam cracking", str(exc))

## H2 route via DAC, PEM, CO2-based methanol, and MTO

In [4]:
try:
    h2_dac = ma.create_DAC()
    add_step("H2 route", "Direct air capture", h2_dac)

    h2_pem = ma.create_H2PEM(avoidedburden="none", eol="no")
    add_step("H2 route", "PEM electrolysis", h2_pem, "modelled without H2 end-of-life")

    h2_methanol = ma.create_CO2hydr(eol="no", carbon_dioxide=h2_dac, hydrogen=h2_pem)
    add_step("H2 route", "CO2 hydrogenation to methanol", h2_methanol, "methanol is an intermediate; no methanol end-of-life")

    h2_ethylene = ma.create_MTO(avoidedburden="none", allocation="weight", eol="no", methanol=h2_methanol)
    add_step(
        "H2 route",
        "Methanol-to-olefins and product/by-product handling",
        h2_ethylene,
        "endpoint is 1 kg produced ethylene; no ethylene use phase or product incineration",
    )
except Exception as exc:
    add_blocker("H2 route", "Route generation", str(exc))

## eCO2R route with detailed separation chain

In [5]:
try:
    eco2r_dac = ma.create_DAC()
    add_step("eCO2R route", "Direct air capture", eco2r_dac)

    eco2r_raw = ma.create_eCO2R(eol="no", carbon_dioxide=eco2r_dac)
    add_step("eCO2R route", "Electrochemical CO2 reduction to ethylene, before separation", eco2r_raw)

    eco2r_vl = ma.create_eCO2R_VL_sep(ethylene_eCO2R=eco2r_raw)
    add_step("eCO2R route", "Vapor-liquid phase separation", eco2r_vl)

    eco2r_deox = ma.create_eCO2R_DeOx_sep(ethylene_VL_sep=eco2r_vl)
    add_step("eCO2R route", "De-Ox separation", eco2r_deox)

    eco2r_amine = ma.create_eCO2R_aminewash_sep(ethylene_DeOx_sep=eco2r_deox)
    add_step("eCO2R route", "Amine wash with CO2 recycle", eco2r_amine, "CO2 recycle is represented inside the amine-wash model")

    eco2r_tsa = ma.create_eCO2R_TSA_sep(ethylene_aminewash_sep=eco2r_amine)
    add_step("eCO2R route", "Temperature swing adsorption and wastewater handling", eco2r_tsa, "wastewater treatment is represented by the TSA model")

    eco2r_ethylene = ma.create_eCO2R_cryo_sep(eol="no", ethylene_TSA_sep=eco2r_tsa)
    add_step(
        "eCO2R route",
        "Cryogenic separation to final ethylene",
        eco2r_ethylene,
        "endpoint is 1 kg produced ethylene; eol=no excludes ethylene product EoL while the separation inventories retain by-product oxidation",
    )
except Exception as exc:
    add_blocker("eCO2R route", "Route generation", str(exc))

## Route activity table

In [6]:
route_table = pd.DataFrame(rows)
display(route_table)

if blockers:
    blocker_table = pd.DataFrame(blockers)
    display(blocker_table)
    raise RuntimeError("One or more required route blocks are missing or failed. See blocker_table above.")

,route,step,activity name,code,database,location,unit,reference product,modelling status
0,Fossil route,"Steam cracking, cradle-to-gate ethylene","unsaturated hydrocarbons production, steam cra...",a14c47e35fef60318b29f28413c6adf0,ecoinvent-3.12-cutoff,RER w/o RU,kilogram,ethylene,existing ecoinvent activity; endpoint is 1 kg ...
1,H2 route,Direct air capture,"direct air capture, 2016","DAC2016|elec=('ecoinvent-3.12-cutoff', '2bf03d...",disco2very,RER,kilogram,"carbon dioxide, in chemical industry",modelled
2,H2 route,PEM electrolysis,"hydrogen production, from PEM water electrolys...",H2PEM|ab=none|eol=no|elec=('ecoinvent-3.12-cut...,disco2very,RER,kilogram,hydrogen,modelled without H2 end-of-life
3,H2 route,CO2 hydrogenation to methanol,"methanol production, from CO2 hydrogenation. e...",CO2hydr | eol=no|elec=('ecoinvent-3.12-cutoff'...,disco2very,RER,kilogram,methanol,methanol is an intermediate; no methanol end-o...
4,H2 route,Methanol-to-olefins and product/by-product han...,"ethylene production, from methanol-to-olefins ...",MTO|ab=none|alloc=weight|eol=no|elec=('ecoinve...,disco2very,RER,kilogram,ethylene,endpoint is 1 kg produced ethylene; no ethylen...
5,eCO2R route,Direct air capture,"direct air capture, 2016","DAC2016|elec=('ecoinvent-3.12-cutoff', '2bf03d...",disco2very,RER,kilogram,"carbon dioxide, in chemical industry",modelled
6,eCO2R route,"Electrochemical CO2 reduction to ethylene, bef...",electrochemical CO2 reduction to ethylene. eol...,eCO2R to ethylene|eol=no|elec=('ecoinvent-3.12...,disco2very,DE,kilogram,ethylene_pre_VL_sep,modelled
7,eCO2R route,Vapor-liquid phase separation,"vapor-liquid separation of ethylene, from eCO2R","eCO2R to ethylene, VL sep|elec=('ecoinvent-3.1...",disco2very,DE,kilogram,ethylene_pre_DeOx,modelled
8,eCO2R route,De-Ox separation,"electrochemical CO2 reduction, oxygen removal ...","eCO2R to ethylene, DeOx sep|elec=('ecoinvent-3...",disco2very,DE,kilogram,ethylene_pre_aminewash,modelled
9,eCO2R route,Amine wash with CO2 recycle,"amine wash separation of ethylene, from eCO2R","eCO2R to ethylene, amine wash sep|heat=('ecoin...",disco2very,DE,kilogram,ethylene_pre_TSA,CO2 recycle is represented inside the amine-wa...
